## Assignment 5: Design Optimization
#### Jahnav Rokalaboina



### Problem Statement: 
A cart runs vertically up and down a wall. Its leg is actuated by a servo mounted in the body at a 
revolute joint. The leg is modeled as a compliant beam, with 2 links and 1 passive joint, and can jum 
by pushing its leg off the groun

Using MuJoCo, find the ideal spring stiffness for this system that permits the body’s center of mass to 
reach the highest point using two approaches.
Use the code and parameters derived in the textbook for modeling an RC servo. You should use t e
final parameters obtained via fittin
g.
Use two approaches to study the effect of stiffness on jump height
1. Sweep through leg stiffness values from 1e-3 to 1e1. Plot the maximum z height of the rob t as
a function of k.
2. program a minimization function to obtain the value that maximizes jump heightd.

In [ ]:
import os
import mujoco
import numpy as np
import mediapy as media
import matplotlib.pyplot as plt
import math

#### Setting up the parameters

In [ ]:
Vnom = 6
R = Vnom/ .6
G = 55.5
t_stall = 15/100/G
i_stall = 0.6
i_nl = 0.2
w_nl = 0.66*1000*2*math.pi/180*G
kt = t_stall/i_stall
ke = kt
b_calc = kt*i_nl/w_nl
ts = 1e-4
V_control = 5
b_fit = 1.404e-6
kp_fit = 8.896


In [ ]:
framerate = 30
data_rate = 100
width = 800
height = 600

#### Defining the XML of the system

In [ ]:
xml_template = '''
<mujoco>
    <option><flag gravity = "enable" contact = "enable"/></option>
    <option timestep ="{ts:e}"/>
    <compiler angle="degree" />
    <visual><global offwidth="{width}" offheight="{height}" /></visual>

    <default>
        <geom contype="1" conaffinity="1" condim="3" friction=".6 .3 .3" solimp=".999 .999 .001" solref=".001 1" margin="0.001" group="0"/>
    </default>


    <worldbody>
        <light name="top" pos="0 0 1"/>
        <body name="floor" pos="0 0 0">
            <geom name="floor" pos="0 0 0" size="1 1 .05" type="plane" rgba="1 .83 .61 .5"/>
        </body>
        <body name="trunk" pos="0 0 .05">
            <joint name="joint1" type="slide" axis="0 0 1"/>
            <geom name="box" pos="0 0 0" size=".025 .025 .025" type ="box" rgba="1 0 0 1" mass=".01"/>
            <body name="leg_link_1" pos="0.025 0 -.025">
                <joint name="joint2" type="hinge" axis="0 1 0" limited="true" range="0 180"/>
                <geom name="leg_link_1" pos="0.025 0 0" size=".025 .0125 .001" type="box" rgba="1 0 0 1" mass=".001"/>
                <body name="leg_link_2" pos="0.05 0 0">
                    <joint name="joint3" type="hinge" axis="0 1 0" stiffness="{k:e}" damping="{b:e}"/>
                    <geom name="leg_link_2" pos="0.025 0 0" size=".025 .0125 .001" type="box" rgba="1 0 0 1" mass=".001"/>
                </body>
            </body>
        </body>
    </worldbody>
    <actuator>
        <motor name="motor1" joint="joint2" />
    </actuator>
</mujoco>
'''

### Defining a fundtion to run simulation taking stiffness and damping as arguements

In [ ]:
def run_sim(k,b,render=False):
    xml = xml_template.format(k =k, b=b, width=width, height = height, ts=ts)
    model = mujoco.MjModel.from_xml_string(xml)
    data = mujoco.MjData(model)
    renderer = mujoco.Renderer(model,width=width,height=height)
    #print(k)
    def my_controller(model, data):
        w = data.qvel[1]
        actual = data.qpos[1]
        
        if data.time>1:
            desired = math.pi
        else:
            desired = 0

        error = desired - actual
        V = kp_fit*error
        if V>V_control: V=V_control
        if V<-V_control: V=-V_control

        torque = (kt*(V-(ke)*w*G)/R - b_fit*w*G)*G
        data.ctrl[0] = torque
        
        return
    try:
        mujoco.set_mjcb_control(my_controller)
        duration = 5
        frames=[]
        t=[]
        xy=[]
        mujoco.mj_resetData(model,data)

        while data.time<duration:
            mujoco.mj_step(model,data)
            if render:
                if len(frames)<data.time*framerate:
                    renderer.update_scene(data)
                    pixels=renderer.render()
                    frames.append(pixels)
            if len(xy)<data.time*data_rate:
                t.append(data.time)
                xy.append(data.xpos.copy())

        mujoco.set_mjcb_control(None)

        if render:
            media.show_video(frames, fps= framerate, width=width, height=height)

        t = np.array(t)
        xy = np.array(xy)
    finally:
        mujoco.set_mjcb_control(None)
    return t,xy,frames


### Video of Arbitrarily selected parameter based simulation result

In [ ]:
t,xy,frames = run_sim(.01,.001,render=True)

#### Plot of height of Center of Mass of body over time for an arbitrarily selected result

In [ ]:
plt.plot(t,xy[:,2,2])
plt.title("Height of Center of Mass of body over time")
plt.xlabel("Time(s)")
plt.ylabel("Height(m)")

#### Plot of stiffness vs max jump height over the range specified above

In [ ]:
zs=[]
ks = []
exps = np.r_[-3:1:.1]
for exp in exps:
    k = 10**(float(exp))
    ks.append(k)
    b = k/100
    t,xy,frmaes = run_sim(k,b,render= False)
    max_z = xy[:,2,2].max()
    zs.append(max_z)

ks=np.array(ks)
zs=np.array(zs)


In [ ]:

plt.semilogx(ks,zs)
plt.title("Stiffness vs max jump height")
plt.xlabel("K")
plt.ylabel("Height(m)")


### Optimization with results overplotted on previous figure

In [ ]:
from scipy.optimize import minimize

# Choosing this range because the system is unstable after index=38 from previous figure
ks_a=ks[0:38]
zs_a=zs[0:38]

# Define the objective function for minimization
def objective(x):
    # Ensure x is within the range of ks
    if x < ks_a.min() or x > ks_a.max():
        return float('inf')  # Penalize out-of-bound inputs
    # Find the closest ks value to x and return the corresponding -zs
    idx = np.argmin(np.abs(ks_a - x))
    return -zs_a[idx]  # Negative to maximize zs

# Perform optimization
result = minimize(objective, x0=ks_a[np.argmax(zs_a)], bounds=[(ks_a.min(), ks_a.max())])

# Extract the results
k_opt = result.x[0]
idx_opt = np.argmin(np.abs(ks_a - k_opt))
z_opt = zs_a[idx_opt]

# Print the results
print(f"Peak found at ks = {k_opt:.5f}, zs = {z_opt:.5f}")
plt.semilogx(ks_a,zs_a)
plt.semilogx(k_opt,z_opt,"*")
plt.axvline(x=k_opt,color="red")
plt.text(k_opt+0.001,z_opt+0.001,"Optimized Max Jump Height")
plt.title("Stiffness vs max jump height")
plt.xlabel("K")
plt.ylabel("Height(m)")

#### Video of the optimized jump.

In [ ]:
t,xy,frames = run_sim(k_opt,k_opt/100,render=True)

####  Plot of height of center of mass of body over time for the optimized design.

In [ ]:
plt.plot(t,xy[:,2,2])
plt.title("Height of Center of Mass of body over time (Optimized)")
plt.xlabel("Time(s)")
plt.ylabel("Height(m)")